# Exploration — Indian Rainfall & Weather Data
Quick exploratory look at the dataset used by the Uncertainty-Aware Rainfall Prediction dashboard.
The production pipeline lives in `src/`; this notebook is for exploration only.

In [ ]:
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '..')
df = pd.read_csv('../data/dataset.csv', parse_dates=['date_of_record'])
df.shape

In [ ]:
df.info()
df.isna().mean().sort_values(ascending=False)

In [ ]:
# Rainfall distribution and the 2.5 mm significance threshold
r = df['rainfall'].dropna()
print('share of days >= 2.5 mm:', (r >= 2.5).mean().round(3))
r.clip(upper=50).hist(bins=50)
plt.axvline(2.5, color='r', ls='--'); plt.title('Daily rainfall (clipped at 50 mm)'); plt.show()

In [ ]:
# Monthly seasonality (monsoon signal)
df.assign(m=df.date_of_record.dt.month).groupby('m').rainfall.mean().plot.bar()
plt.title('Mean daily rainfall by month'); plt.show()

In [ ]:
# Run the production preprocessing + feature pipeline on a sample
from src.config import AppConfig
from src.data_loader import auto_map_columns
from src.preprocessing import preprocess
from src.feature_engineering import build_features
cfg = AppConfig()
sample = df[df.station_name.isin(df.station_name.unique()[:20])]
mapping = auto_map_columns(sample)
clean, prep_report = preprocess(sample, mapping, cfg)
feats, cols, feat_report = build_features(clean, cfg)
prep_report, feat_report['positive_rate']

In [ ]:
# Autocorrelation of rain days — persistence is the key predictive signal
one = feats[feats.station == feats.station.iloc[0]].set_index('date')
pd.plotting.autocorrelation_plot(one['rain_today_flag'].iloc[:1000])
plt.xlim(0, 30); plt.title('Rain-day autocorrelation (first station)'); plt.show()